# Optional agent alignment and prompt optimization lab

This connected, disabled-by-default lab adapts MLflow's [agent alignment and optimization cookbook](https://mlflow.org/cookbook/agent-alignment-optimization/) to the repository's stricter release lifecycle.

The safe sequence is:

1. calibrate a domain judge against human feedback;
2. validate the frozen judge on held-out labels;
3. optimize an exact seed prompt against a separate training split and bounded request budget;
4. register the optimized text as a new immutable prompt version;
5. evaluate that exact version on a final held-out release split;
6. let the ordinary release gate decide `adopt`, `reject`, or `inconclusive`.

This notebook never moves `production`. Optimization proposes a change; it does not authorize deployment.

In [ ]:
import sys
from pathlib import Path

repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "examples" / "support").is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open the cloned repository as your workspace.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 1. Lock three disjoint evidence splits

Judge calibration, optimizer training, and final release testing must not reuse cases. Otherwise the aligned judge and optimized prompt can certify the same examples they learned from.

In [ ]:
from examples.support.optimization import split_contract_summary

split_contract = split_contract_summary()
print(f"split manifest digest: {split_contract['split_manifest_digest']}")
for name, count in split_contract["case_counts"].items():
    print(f"{name}: {count} disjoint cases")

## 2. Optimize on the training split, prove on the held-out split

A miniature offline optimizer makes the contract concrete: one seed instruction plus three fixed prompt edits — a citation requirement, a brevity rule, and pasted worked examples copied from the training cases. A deterministic toy generator turns each prompt variant into an answer whose text visibly reflects that variant's rules, and the same deterministic fact, citation, and policy scorers from the cost lesson grade every variant on `optimizer_training` only. The winner then earns exactly one evaluation on `held_out_release`.

In [ ]:
from examples.support.optimization import run_toy_prompt_optimization

toy_optimization = run_toy_prompt_optimization()
for name, score in toy_optimization["train_scores"].items():
    print(f"train score {name}: {score:.3f}")
print(f"winner on optimizer_training only: {toy_optimization['winner']}")

In [ ]:
from examples.support.optimization import evaluate_winner_on_holdout

heldout_evidence = evaluate_winner_on_holdout(toy_optimization)
print(f"winner: {heldout_evidence['winner']}")
print(f"train score: {heldout_evidence['train_score']:.3f}")
print(f"holdout score: {heldout_evidence['holdout_score']:.3f}")
print(f"generalization gap: {heldout_evidence['generalization_gap']:.3f}")
print(
    "The winning edit pasted training answers into the prompt; disjoint "
    "held-out wording exposes that it learned the cases, not the task."
)

## 3. Show why the splits must not overlap

Rerun the same winner, but grade it on a leaked "holdout" that reuses the optimizer-training cases. The leaked estimate reports the memorized score and hides the failure entirely — the inflation below is the reason judge calibration, optimizer training, and release testing must never share cases.

In [ ]:
from examples.support.optimization import demonstrate_split_leakage

leakage = demonstrate_split_leakage(toy_optimization)
print(f"honest disjoint holdout: {leakage['honest_disjoint_holdout']:.3f}")
print(f"leaked overlapping holdout: {leakage['leaked_overlapping_holdout']:.3f}")
print(f"leak inflation: +{leakage['leak_inflation']:.3f}")
print(
    "An overlapping split would have certified the memorizing prompt for "
    "release; the disjoint split contract is what catches it."
)

## 4. Make experimental dependencies and spend bounds explicit

`MemAlignOptimizer` is experimental and requires DSPy. `GepaPromptOptimizer` requires GEPA. Neither dependency is present in this repository's certified locks, so the connected lab must remain off unless dependency policy, exact locks, template locks, and compatibility evidence are updated together.

In [ ]:
from examples.support.optimization import experimental_dependency_status

dependency_status = experimental_dependency_status()
EXPERIMENTAL_DEPENDENCIES = dependency_status["experimental_dependencies"]
for key, value in dependency_status.items():
    print(f"{key}: {value}")

## 5. Connected alignment and optimization skeleton

Before enabling the next cell:

- collect at least the approved number of balanced human labels with rationales and source `group:domain-reviewers`;
- use the same assessment name for human feedback and the judge;
- freeze and validate the aligned judge before optimization;
- set the exact seed prompt, optimizer data, governed reflection-model URI, and validated judge version/run evidence;
- make `predict_fn` load and format the registered seed prompt **inside every call**. Building an agent once around the baseline text would leave every optimizer proposal inert.

In [ ]:
from examples.support.optimization import AlignmentConfig, run_alignment_workflow

PERSIST_EVIDENCE_TO_DATABRICKS = False
RUN_EXPERIMENTAL_OPTIMIZATION = False
JUDGE_NAME = "uncertainty_explanation"
JUDGE_EXPERIMENT_ID = SEED_PROMPT_URI = REFLECTION_MODEL_URI = None
ALIGNED_JUDGE_VERSION = JUDGE_VALIDATION_RUN_ID = None
JUDGE_VALIDATION_AGREEMENT = JUDGE_VALIDATION_LABEL_COUNT = None

alignment_config = AlignmentConfig(
    persist_evidence=PERSIST_EVIDENCE_TO_DATABRICKS,
    run_optimization=RUN_EXPERIMENTAL_OPTIMIZATION,
    judge_name=JUDGE_NAME,
    judge_experiment_id=JUDGE_EXPERIMENT_ID,
    seed_prompt_uri=SEED_PROMPT_URI,
    reflection_model_uri=REFLECTION_MODEL_URI,
    aligned_judge_version=ALIGNED_JUDGE_VERSION,
    judge_validation_run_id=JUDGE_VALIDATION_RUN_ID,
    judge_validation_agreement=JUDGE_VALIDATION_AGREEMENT,
    judge_validation_label_count=JUDGE_VALIDATION_LABEL_COUNT,
)
if PERSIST_EVIDENCE_TO_DATABRICKS or RUN_EXPERIMENTAL_OPTIMIZATION:
    print(run_alignment_workflow(alignment_config))
else:
    print("DATABRICKS EVIDENCE AND EXPERIMENTAL OPTIMIZATION SKIPPED")

## 6. Register, test held-out cases, then use the normal gate

If optimization produces a useful template:

1. register it as a new immutable version of the same qualified prompt;
2. record its content digest, exact URI, seed URI, split-manifest digest, optimizer/reflection model, dependency digest, request budget, and source commit;
3. load that exact new version inside every prediction on `held_out_release`;
4. run deterministic fact, citation, policy, critical-row, latency, token, cost, and cost-coverage gates, with the validated judge as additional evidence;
5. compare against the untouched baseline and choose `adopt`, `reject`, or `inconclusive`;
6. move a controlled alias only in the ordinary release workflow after all checks pass.

Do not inspect private judge fields such as `_semantic_memory`; record public versioned instructions and alignment evidence.

In [ ]:
from examples.support.optimization import optimization_plan

for key, value in optimization_plan().items():
    print(f"{key}: {value}")
print(
    "Optimization evidence is not release evidence: the decision stays "
    "'inconclusive' and release stays blocked until the ordinary gate runs."
)

## 5. The judge is an instrument: measure it inside the run

A baseline-versus-change delta is a statement about the agent only while the
judge held still. Two per-run checks make that assumption measurable instead
of hoped for:

- **Self-consistency** re-judges a small deterministic sample of the run's
  own outputs. The judge's disagreement with itself bounds how much of any
  delta is signal.
- **Anchor drift** re-judges frozen baseline outputs whose scores were
  recorded when the baseline was established (`evals/judge_anchors.json`,
  digest-bound). The agent is not in that loop, so movement here is the
  judge changing — a provider-side repoint reads as drift, never as an
  agent regression.

Below, the same recorded answers are re-scored by a steady judge and then by
a "silently repointed" one. The drifted judge moves every frozen anchor while
the agent outputs never changed, and the reading says which artefact to
debug. In a generated project these checks run inside every judged
`agentkit compare` via the `integrity:` block, and `require_anchors: true`
turns the drift reading into a gate rule.

In [ ]:
from examples.support.optimization import judge_stability_summary

judge_stability_summary()

## 6. Commit the calibration record the gate can demand

"Our agent scores 0.87 on correctness" is unfalsifiable. The auditable claim
is "0.87 under a judge that agrees with our reviewers at kappa 0.69 on a
named label set, against a human ceiling of 0.75" — chance-adjusted
(Cohen's kappa), because raw percent agreement is inflated by class
imbalance, and bounded by the ceiling, because a judge cannot be more
consistent than the humans who define the target. A low ceiling means the
rubric is the problem: fix the rubric before touching the model.

The labels below are the calibration split locked in section 1, labelled by
two fictional reviewer groups. Ties carry no target for the judge to match,
so they are excluded — and counted, because many ties is a rubric finding,
not noise to discard. In a project, `agentkit judge calibrate` computes
exactly this and writes the committed record `evals/judges/<scorer>.json`;
setting `integrity.require_calibration: true` then makes every judged run
and the promotion gate demand a passing, version-matched record.

In [ ]:
from examples.support.optimization import judge_calibration_summary

judge_calibration_summary()